# Basic Imports

In [ ]:
import json
import uuid
from openai import OpenAI


# Defining Function and Menu

In [ ]:
import uuid  # Import uuid module for generating unique order IDs

# Menu dictionary with items, their price, stock quantity, and description
menu = {
    "burger": {"price": 150, "stock": 10, "description": "Delicious beef burger"},
    "pizza": {"price": 300, "stock": 5, "description": "Cheesy pepperoni pizza"},
    "pasta": {"price": 250, "stock": 8, "description": "Creamy alfredo pasta"},
    "coke": {"price": 50, "stock": 20, "description": "Refreshing soft drink"},
    "sandwich": {"price": 120, "stock": 15, "description": "Grilled cheese sandwich"},
    "fries": {"price": 100, "stock": 12, "description": "Crispy golden french fries"},
    "mojito": {"price": 180, "stock": 10, "description": "Cool mint mojito"},
    "coffee": {"price": 120, "stock": 20, "description": "Hot brewed coffee"},
    "tea": {"price": 80, "stock": 25, "description": "Refreshing herbal tea"}
}

# Cart to hold items user wants to buy
cart = {}

# Order history to store past orders with order IDs
orderHistory = {}

def getMenu():
    # Return the current menu as a string
    return str(menu)

def addToCart(item, quantity):
    # Add given quantity of item to cart if available in stock
    item = item.lower()
    if item in menu:
        if menu[item]["stock"] >= quantity:
            cart[item] = cart.get(item, 0) + quantity  # Increase quantity in cart
            menu[item]["stock"] -= quantity  # Reduce stock accordingly
            return str({"message": f"{quantity} {item}(s) added to cart.", "cart": cart})
        else:
            # Not enough stock available
            return str({"error": f"Only {menu[item]['stock']} {item}(s) available."})
    # Item not found in menu
    return str({"error": "Item not available in menu."})

def removeFromCart(item, quantity):
    # Remove given quantity of item from cart, update stock accordingly
    item = item.lower()
    if item in cart:
        if cart[item] > quantity:
            cart[item] -= quantity  # Decrease quantity in cart
            menu[item]["stock"] += quantity  # Increase stock accordingly
            return str({"message": f"{quantity} {item}(s) removed from cart.", "cart": cart})
        else:
            # Remove item completely if quantity to remove >= in cart
            menu[item]["stock"] += cart[item]  # Restore full quantity to stock
            del cart[item]
            return str({"message": f"{item} removed from cart.", "cart": cart})
    # Item not found in cart
    return str({"error": "Item not in cart."})

def getOrderDetails():
    # Generate order details and order ID, clear cart after placing order
    if not cart:
        return str({"message": "Your cart is empty."})
    # Calculate total cost based on price and quantity
    total = sum(menu[item]["price"] * qty for item, qty in cart.items())
    # Generate a short unique order ID
    order_id = str(uuid.uuid4())[:8]
    # Store order details in orderHistory
    orderHistory[order_id] = {"cart": cart.copy(), "total": total}
    cart.clear()  # Clear cart after order placed
    return str({"orderId": order_id, "order": orderHistory[order_id]})

def clearCart():
    # Clear cart and restore stock for all items in cart
    for item, qty in cart.items():
        menu[item]["stock"] += qty
    cart.clear()
    return str({"message": "Cart has been cleared."})

def viewOrderHistory():
    # Return past orders or message if no past orders found
    return str(orderHistory) if orderHistory else str({"message": "No past orders."})


## Defining tools

In [ ]:
tools = [
    {"type": "function", "function": {"name": "getMenu", "description": "Get the restaurant menu.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "addToCart", "description": "Add an item to the cart.", "parameters": {"type": "object", "properties": {"item": {"type": "string"}, "quantity": {"type": "integer"}}, "required": ["item", "quantity"], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "removeFromCart", "description": "Remove an item from the cart.", "parameters": {"type": "object", "properties": {"item": {"type": "string"}, "quantity": {"type": "integer"}}, "required": ["item", "quantity"], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "getOrderDetails", "description": "Get the order details and generate an order ID.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "clearCart", "description": "Clear all items from the cart.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "viewOrderHistory", "description": "View past order history.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}}
]

# Setting Api key

In [ ]:
key ="Your Api Key"
client = OpenAI(api_key=key)

In [ ]:
import json
import pandas as pd

## Function to execute the tool call based on the tool name and its arguments
def executeToolCall(toolCall):
    tool = toolCall.function          # Extract the tool/function called by the agent
    args = json.loads(tool.arguments) # Parse the JSON string of arguments into a dictionary

    # Call the corresponding function based on the tool name
    if tool.name == "getMenu":
        return getMenu()

    if tool.name == "addToCart":
        return addToCart(args["item"], args["quantity"])

    if tool.name == "removeFromCart":
        return removeFromCart(args["item"], args["quantity"])

    if tool.name == "getOrderDetails":
        return getOrderDetails()

    if tool.name == "clearCart":
        return clearCart()

    if tool.name == "viewOrderHistory":
        return viewOrderHistory()

    # Return this if the tool name is not recognized
    return "Unknown tool call."

# List of user queries to simulate conversation and interaction with the agent
user_queries = [
    "Show me the menu",
    "Add 2 burgers to my cart",
    "Add 1 coke to my cart",
    "Remove 1 burger from my cart",
    "Place my order"
]

# List to hold the results for each user query
results = []

# Loop through each query from the user
for query in user_queries:
    # Start the message list with the user's input
    messages = [{"role": "user", "content": query}]

    # Call the chat completion API with the current messages and available tools
    response = client.chat.completions.create(model="gpt-4o", messages=messages, tools=tools)

    # Get the AI's initial response message
    aiMessage = response.choices[0].message

    # Append the AI message to the message history
    messages.append(aiMessage)

    output = ""

    # Check if the AI has decided to call any tools
    if aiMessage.tool_calls:
        # For each tool call requested by the AI
        for toolCall in aiMessage.tool_calls:
            # Execute the tool function and get its response
            toolResponse = executeToolCall(toolCall)

            # Append the tool's response to the message history with tool call ID
            messages.append({"role": "tool", "content": toolResponse, "tool_call_id": toolCall.id})

        # After tool responses, call the chat model again to get final response using updated message history
        finalResponse = client.chat.completions.create(model="gpt-4o", messages=messages, tools=tools)

        # Extract the final AI response content
        output = finalResponse.choices[0].message.content
    else:
        # If no tools were called, just use the AI's initial content response
        output = aiMessage.content

    # Save all relevant info for this query in the results list
    results.append({
        "user_query": query,
        # Save the message history as a pretty-printed JSON string
        "message_history": json.dumps(
            [m.model_dump() if hasattr(m, "model_dump") else {"role": m["role"], "content": m["content"]} for m in messages],
            indent=2
        ),
        "output": output,
        # Save all tool descriptions (for reference or debugging)
        "all_tool_descriptions": json.dumps([t['function']['description'] for t in tools], indent=2)
    })

# Convert the results list into a pandas DataFrame for easy viewing and analysis
df = pd.DataFrame(results)

# Print first few rows of the DataFrame to check outputs
print(df.head())


                     user_query  \
0              Show me the menu   
1      Add 2 burgers to my cart   
2         Add 1 coke to my cart   
3  Remove 1 burger from my cart   
4                Place my order   

                                     message_history  \
0  [\n  {\n    "role": "user",\n    "content": "S...   
1  [\n  {\n    "role": "user",\n    "content": "A...   
2  [\n  {\n    "role": "user",\n    "content": "A...   
3  [\n  {\n    "role": "user",\n    "content": "R...   
4  [\n  {\n    "role": "user",\n    "content": "P...   

                                              output  \
0  Here's the menu with descriptions and prices:\...   
1                 I've added 2 burgers to your cart.   
2  1 coke has been added to your cart. Your curre...   
3  1 burger has been removed from your cart. Now,...   
4  Your current order contains the following item...   

                               all_tool_descriptions  
0  [\n  "Get the restaurant menu.",\n  "Add an it...  
1  [\

In [ ]:
# raw dataframe
df


,user_query,message_history,output,all_tool_descriptions
0,Show me the menu,"[\n {\n ""role"": ""user"",\n ""content"": ""S...",Here's the menu with descriptions and prices:\...,"[\n ""Get the restaurant menu."",\n ""Add an it..."
1,Add 2 burgers to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",I've added 2 burgers to your cart.,"[\n ""Get the restaurant menu."",\n ""Add an it..."
2,Add 1 coke to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",1 coke has been added to your cart. Your curre...,"[\n ""Get the restaurant menu."",\n ""Add an it..."
3,Remove 1 burger from my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""R...","1 burger has been removed from your cart. Now,...","[\n ""Get the restaurant menu."",\n ""Add an it..."
4,Place my order,"[\n {\n ""role"": ""user"",\n ""content"": ""P...",Your current order contains the following item...,"[\n ""Get the restaurant menu."",\n ""Add an it..."


# Evaluate Using LLumo

In [ ]:
import pandas as pd

# Read the CSV file named 'openaiResults.csv' into a DataFrame
df = pd.read_csv("openaiResults.csv")

# Rename columns for better readability or consistency
df.rename(columns={
    "user_query": "query",
    "message_history": "messageHistory",
    "all_tool_descriptions": "tools"
}, inplace=True)

# Display the resulting DataFrame
print(df)


,query,messageHistory,output,tools
0,Show me the menu,"[\n {\n ""role"": ""user"",\n ""content"": ""S...",Here's the menu with descriptions and prices:\...,"[\n ""Get the restaurant menu."",\n ""Add an it..."
1,Add 2 burgers to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",I've added 2 burgers to your cart.,"[\n ""Get the restaurant menu."",\n ""Add an it..."
2,Add 1 coke to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",1 coke has been added to your cart. Your curre...,"[\n ""Get the restaurant menu."",\n ""Add an it..."
3,Remove 1 burger from my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""R...","1 burger has been removed from your cart. Now,...","[\n ""Get the restaurant menu."",\n ""Add an it..."
4,Place my order,"[\n {\n ""role"": ""user"",\n ""content"": ""P...",Your current order contains the following item...,"[\n ""Get the restaurant menu."",\n ""Add an it..."


# Install required package

In [ ]:
!pip install llumo

In [ ]:
from llumo import LlumoClient

# Initialize the LlumoClient with your API key
client = LlumoClient(api_key="key_ZGZjMTZmZjk5Y2M1NWQ1YzU2NjhmMzVj_32e0ed987bf5b9fce05bf8bdd45f427f43b0099b578a621c26e52815b3018a762171f342c87df6a1df421c7c16f523b8633811db2a9e583d6a9e88a91952b9da7d79638966ed2e900f678696903f45027f9e4a3ead9e4e65caf3546ce046d9f4e8b0322b811915d477698d3b5a2fa25026ae0b87a7d8157ca474132b7666f663")

# Use the client to evaluate agent responses based on the provided DataFrame 'df'
# 'prompt_template' is used to format the prompt for each query dynamically
result = client.evaluateAgentResponses(
    dataframe=df,
    prompt_template="Provide answer for the given query: {{query}}"
)




======= Running evaluation for: Tool Reliability =======

======= Running evaluation for: Stepwise Progression =======

======= Running evaluation for: Tool Selection Accuracy =======

======= Running evaluation for: Final Task Alignment =======


In [ ]:
result

,query,messageHistory,output,tools,Tool Reliability,Tool Reliability Reason,Stepwise Progression,Stepwise Progression Reason,Tool Selection Accuracy,Tool Selection Accuracy Reason,Final Task Alignment,Final Task Alignment Reason
0,Show me the menu,"[\n {\n ""role"": ""user"",\n ""content"": ""S...",Here's the menu with descriptions and prices:\...,"[\n ""Get the restaurant menu."",\n ""Add an it...",100,The `getMenu` tool successfully executed and r...,100,The tool 'getMenu' is relevant to the user que...,100,The user requested the menu. The assistant use...,99,The assistant successfully retrieved and displ...
1,Add 2 burgers to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",I've added 2 burgers to your cart.,"[\n ""Get the restaurant menu."",\n ""Add an it...",100,The `addToCart` tool successfully added 2 burg...,99,The tool call `addToCart` is relevant to the u...,100,"The assistant used only the ""Add an item to th...",100,The assistant added 2 burgers to the cart as r...
2,Add 1 coke to my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""A...",1 coke has been added to your cart. Your curre...,"[\n ""Get the restaurant menu."",\n ""Add an it...",99,The `addToCart` tool successfully added the co...,100,The tool 'addToCart' is relevant to the user q...,100,"The assistant used only the ""Add an item to th...",100,The user requested to add a coke to their cart...
3,Remove 1 burger from my cart,"[\n {\n ""role"": ""user"",\n ""content"": ""R...","1 burger has been removed from your cart. Now,...","[\n ""Get the restaurant menu."",\n ""Add an it...",100,The `removeFromCart` tool successfully removed...,99,The tool call `removeFromCart` directly addres...,99,The assistant used only the 'Remove an item fr...,100,The user requested removal of a burger from th...
4,Place my order,"[\n {\n ""role"": ""user"",\n ""content"": ""P...",Your current order contains the following item...,"[\n ""Get the restaurant menu."",\n ""Add an it...",100,The getOrderDetails tool successfully executed...,1,No tools were used to add items to the cart be...,2,"The assistant used the tool ""getOrderDetails"",...",1,The user requested to place an order. The assi...
